### Step 1. Create a config file and store it in Volume

In [0]:
config_path = "/Volumes/workspace/default/finance_volume/config/ingestion_config.json"

config_json = """
[
  {
    "ingestion_name": "finance_transactions",
    "is_active": true,
    "source": {
      "path": "/Volumes/workspace/default/finance_volume/finance_transactions_10mb.csv",
      "format": "csv",
      "options": {
        "header": true,
        "inferSchema": true
      }
    },
    "layers": {
      "bronze": {
        "catalog": "workspace",
        "schema": "bronze",
        "table": "finance_transactions_bronze",
        "write_mode": "overwrite"
      },
      "silver": {
        "catalog": "workspace",
        "schema": "silver",
        "table": "finance_transactions_silver",
        "write_mode": "overwrite"
      },
      "gold": {
        "catalog": "workspace",
        "schema": "gold",
        "table": "finance_transactions_gold",
        "write_mode": "overwrite"
      }
    }
  }
]
"""

dbutils.fs.mkdirs("/Volumes/workspace/default/finance_volume/config")
dbutils.fs.put(config_path, config_json, overwrite=True)

print(f"Config file created at: {config_path}")

### Step 2. See the file content

In [0]:
#print(dbutils.fs.head(config_path))

### Step 3. Load the Configuration

In [0]:
import json

config_path = "/Volumes/workspace/default/finance_volume/config/ingestion_config.json"
ingestion_name = "finance_transactions"


def load_ingestion_config():
    with open(config_path, "r") as file:
        configs = json.load(file)

    for config in configs:
        if config["ingestion_name"] == ingestion_name and config["is_active"] == True:
            return config

    raise Exception(f"No active config found for ingestion name: {ingestion_name}")


def get_layer_details(config, layer_name):
    layer = config["layers"][layer_name]

    layer["full_table_name"] = (
        f"{layer['catalog']}.{layer['schema']}.{layer['table']}"
    )

    return layer


config = load_ingestion_config()

source = config["source"]

bronze = get_layer_details(config, "bronze")
silver = get_layer_details(config, "silver")
gold = get_layer_details(config, "gold")

print("Config loaded successfully")
print(f"Bronze table: {bronze['full_table_name']}")
print(f"Silver table : {silver['full_table_name']}")
print(f"Gold table   : {gold['full_table_name']}")